# 資料處理

## 檢查版本

In [20]:
import sys, xarray as xr
print("Python exe:", sys.executable)
print("Engines:", xr.backends.list_engines())

Python exe: /home/jundian/miniconda3/envs/geospatial-neural-adapter/bin/python
Engines: {'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'h5netcdf': <H5netcdfBackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


## 讀nc4

In [21]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:12<00:00, 42.18it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

In [22]:
# ===== 只看美國本土範圍，並從中抽樣 25 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180（如果本來就是 -180~180，這個轉換也不會壞掉）
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍（可以再微調）
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([lon_all_180[idx_us_mainland], lat_all[idx_us_mainland]])  # (n_us, 2)

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# 從美國本土格點中隨機抽樣 25 個（如果不足 25，就全部使用）
n_sample = min(25, len(idx_us_mainland))
np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(len(idx_us_mainland), size=n_sample, replace=False)

# 這是「在原始格點編號中的 index」
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後的 time series 與座標
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([lon_all_180[sample_idx_global], lat_all[sample_idx_global]])  # (n_sample, 2)

# 如果之後畫圖想分開拿 lon / lat：
lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")


美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 25 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


## Full time

In [23]:
# ===== VAR on the 25 sampled US grid cells (full time range, seasonal part handled manually) =====
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VAR baseline on 25 sampled US grid cells (non-seasonal part only, full time range) =====")

# --------------------------------------------------
# 0. 基本設定與「原始」ts_df_raw
# --------------------------------------------------
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 25
T_total_raw = y_sample.shape[1]

ts_df_raw = pd.DataFrame(
    y_sample.T,  # (T_raw, 25)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps (raw):", T_total_raw)
print("ts_df_raw shape:", ts_df_raw.shape)  # (T_raw, 25)

# --------------------------------------------------
# 1. 手動處理季節成分：做季節差分（lag = 12）
#    這裡假設是月均資料，季節週期為 12 個月
# --------------------------------------------------
SEASONAL_LAG = 12  # 一年 12 個月

ts_df = ts_df_raw.diff(SEASONAL_LAG).dropna()  # (T_total, 25)，已去掉前 12 期
T_total = ts_df.shape[0]
idx_time = ts_df.index

print("\nAfter seasonal differencing (lag = 12):")
print("Total time steps (seasonally differenced):", T_total)
print("ts_df shape (seasonally differenced):", ts_df.shape)

# --------------------------------------------------
# 2. 切成 70 / 15 / 15（train / val / test）
# --------------------------------------------------
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

print(f"Train range: {idx_time[0]} → {split_1_time}")
print(f"Val   range: {split_1_time} → {split_2_time}")
print(f"Test  range: {split_2_time} → {idx_time[-1]}")

# --------------------------------------------------
# 3. 用 train 統計量標準化（針對「季節差分後」的數列）
# --------------------------------------------------
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec

T_train = cut_train
T_val   = cut_val - cut_train
T_test  = T_total - cut_val

print("\n[Scaled series lengths (non-seasonal)]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（季節差分後、原單位），用來算 MSE/MAE/R²（non-seasonal）
vals_all      = ts_df.to_numpy(dtype=np.float32)     # (T_total, 25) 差分後
raw_all       = ts_df_raw.to_numpy(dtype=np.float32) # (T_total_raw, 25) 原始 level

val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val, 25)
test_true_raw = vals_all[cut_val:, :]             # (T_test, 25)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳「季節差分後」序列的反標準化值，shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 自己 loop VAR(p)，建立 AIC / BIC / HQIC / FPE table，並選階
#      注意：這裡是對「季節差分後」的 train_scaled_df 做 VAR(p)
# ======================================================
print("\n[Order selection by manual VAR(p) grid on non-seasonal series, "
      "with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)，可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK: AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, "
              f"HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（{e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 在 table 中標出每個指標的最佳（最小）值 ----
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"]],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"]],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"]],
})

printed_table.loc[best_idx_aic,  "AIC"]  += "*"
printed_table.loc[best_idx_bic,  "BIC"]  += "*"
printed_table.loc[best_idx_hqic, "HQIC"] += "*"
printed_table.loc[best_idx_fpe,  "FPE"]  += "*"

print("\nInformation Criteria table (p = 1..{} on non-seasonal series):"
      .format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 用 BIC 選 lag
p_selected = int(ic_table.loc[best_idx_bic, "lag"])
print(f"\nSelected lag by BIC = {p_selected}")
print(f"Final VAR ORDER (non-seasonal) = ({p_selected})")

# ======================================================
# 4. 用 statsmodels VAR(p_selected) 做預測（季節差分 + 標準化後）
# ======================================================
print(f"\n[Step] VAR(p={p_selected}) via statsmodels on non-seasonal *scaled* series")

# 4.1 建立 train / val / test 的 scaled DataFrame
ts_df_scaled_full = ts_df_scaled.astype("float64").copy()

train_scaled_df = ts_df_scaled_full.iloc[:cut_train]
val_scaled_df   = ts_df_scaled_full.iloc[cut_train:cut_val]
test_scaled_df  = ts_df_scaled_full.iloc[cut_val:]

T_train = train_scaled_df.shape[0]
T_val   = val_scaled_df.shape[0]
T_test  = test_scaled_df.shape[0]

print("Check lengths from scaled DF:",
      "T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 4.2 train → predict VAL（在「季節差分 + 標準化」空間）
print("\n[Step] VAR: train → predict VAL (non-seasonal, scaled)")

var_train = VAR(train_scaled_df)
res_train = var_train.fit(p_selected)

# statsmodels VAR.forecast 需要最後 p_selected 個觀測值當作起點
y0_val = train_scaled_df.values[-p_selected:, :]      # shape (p_selected, 25)
pred_val_scaled = res_train.forecast(y0_val, steps=T_val)  # (T_val, 25)

# 反標準化回到「季節差分後」的原單位（仍然是差分）
pred_val_raw = inverse_scale(pred_val_scaled)         # (T_val, 25)

# 4.3 train+val → predict TEST（在「季節差分 + 標準化」空間）
print("[Step] VAR: train+val → predict TEST (non-seasonal, scaled)")

train_val_scaled_df = ts_df_scaled_full.iloc[:cut_val]
var_train_val = VAR(train_val_scaled_df)
res_train_val = var_train_val.fit(p_selected)

y0_test = train_val_scaled_df.values[-p_selected:, :]     # shape (p_selected, 25)
pred_test_scaled = res_train_val.forecast(y0_test, steps=T_test)  # (T_test, 25)

pred_test_raw = inverse_scale(pred_test_scaled)           # (T_test, 25)

# ======================================================
# 5. 差分空間上的 VAL / TEST MSE / MAE / RMSE / R²（non-seasonal）
# ======================================================
y_val_true_diff  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred_diff  = pred_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true_diff = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred_diff = pred_test_raw.reshape(-1, SAMPLE_SIZE)

mse_var_val_diff  = mean_squared_error(y_val_true_diff,  y_val_pred_diff)
mae_var_val_diff  = mean_absolute_error(y_val_true_diff, y_val_pred_diff)
mse_var_test_diff = mean_squared_error(y_test_true_diff, y_test_pred_diff)
mae_var_test_diff = mean_absolute_error(y_test_true_diff, y_test_pred_diff)

rmse_var_val_diff  = np.sqrt(mse_var_val_diff)
rmse_var_test_diff = np.sqrt(mse_var_test_diff)

r2_var_val_diff  = r2_score(y_val_true_diff,  y_val_pred_diff)
r2_var_test_diff = r2_score(y_test_true_diff, y_test_pred_diff)

print("\n===== VAR RESULTS on DIFFERENCED series (25 cells, statsmodels VAR) =====")
print(f"[DIFF] VAL  RMSE: {rmse_var_val_diff:.4f}, MSE: {mse_var_val_diff:.4f}, "
      f"MAE: {mae_var_val_diff:.4f}, R²: {r2_var_val_diff:.4f}")
print(f"[DIFF] TEST RMSE: {rmse_var_test_diff:.4f}, MSE: {mse_var_test_diff:.4f}, "
      f"MAE: {mae_var_test_diff:.4f}, R²: {r2_var_test_diff:.4f}")

# ======================================================
# 5.5 把季節差分預測「積回」原始溫度，在原始 level 上算 VAL / TEST 指標
# ======================================================
# 差分的第 k 列 (0-based, 在 ts_df 中) 對應 ts_df_raw 的第 (k + SEASONAL_LAG) 列：
#   diff_k = raw[k + SEASONAL_LAG] - raw[k]
# => 預測 level: y_hat[k + SEASONAL_LAG] = diff_hat[k] + raw[k]

# 對 VAL：
k_val = np.arange(cut_train, cut_val)  # 差分上的 index
# 真實原始溫度（VAL 對應的 level）
y_val_true_level = raw_all[k_val + SEASONAL_LAG, :]   # shape (T_val, 25)
# 季節 lag 的原始溫度 y_{t-12}
y_val_lag_level  = raw_all[k_val, :]                  # shape (T_val, 25)
# 預測原始溫度 = 預測差分 + lag level
y_val_pred_level = pred_val_raw + y_val_lag_level     # shape (T_val, 25)

# 對 TEST：
k_test = np.arange(cut_val, T_total)
y_test_true_level = raw_all[k_test + SEASONAL_LAG, :]  # shape (T_test, 25)
y_test_lag_level  = raw_all[k_test, :]                 # shape (T_test, 25)
y_test_pred_level = pred_test_raw + y_test_lag_level   # shape (T_test, 25)

# level 空間上扁平後算指標
y_val_true_lvl_flat  = y_val_true_level.reshape(-1, SAMPLE_SIZE)
y_val_pred_lvl_flat  = y_val_pred_level.reshape(-1, SAMPLE_SIZE)
y_test_true_lvl_flat = y_test_true_level.reshape(-1, SAMPLE_SIZE)
y_test_pred_lvl_flat = y_test_pred_level.reshape(-1, SAMPLE_SIZE)

mse_var_val_lvl  = mean_squared_error(y_val_true_lvl_flat,  y_val_pred_lvl_flat)
mae_var_val_lvl  = mean_absolute_error(y_val_true_lvl_flat, y_val_pred_lvl_flat)
mse_var_test_lvl = mean_squared_error(y_test_true_lvl_flat, y_test_pred_lvl_flat)
mae_var_test_lvl = mean_absolute_error(y_test_true_lvl_flat, y_test_pred_lvl_flat)

rmse_var_val_lvl  = np.sqrt(mse_var_val_lvl)
rmse_var_test_lvl = np.sqrt(mse_var_test_lvl)

r2_var_val_lvl  = r2_score(y_val_true_lvl_flat,  y_val_pred_lvl_flat)
r2_var_test_lvl = r2_score(y_test_true_lvl_flat, y_test_pred_lvl_flat)

print("\n===== VAR RESULTS on ORIGINAL LEVEL (25 cells, reconstructed from seasonal diff) =====")
print(f"[LEVEL] VAL  RMSE: {rmse_var_val_lvl:.4f}, MSE: {mse_var_val_lvl:.4f}, "
      f"MAE: {mae_var_val_lvl:.4f}, R²: {r2_var_val_lvl:.4f}")
print(f"[LEVEL] TEST RMSE: {rmse_var_test_lvl:.4f}, MSE: {mse_var_test_lvl:.4f}, "
      f"MAE: {mae_var_test_lvl:.4f}, R²: {r2_var_test_lvl:.4f}")

metrics_table_var = pd.DataFrame({
    "Model": ["VAR_nonseasonal_diff", "VAR_nonseasonal_diff",
              "VAR_level_reconstructed", "VAR_level_reconstructed"],
    "Split": ["VAL",                  "TEST",
              "VAL",                  "TEST"],
    "RMSE":  [rmse_var_val_diff,  rmse_var_test_diff,
              rmse_var_val_lvl,   rmse_var_test_lvl],
    "MSE":   [mse_var_val_diff,   mse_var_test_diff,
              mse_var_val_lvl,    mse_var_test_lvl],
    "MAE":   [mae_var_val_diff,   mae_var_test_diff,
              mae_var_val_lvl,    mae_var_test_lvl],
    "R2":    [r2_var_val_diff,    r2_var_test_diff,
              r2_var_val_lvl,     r2_var_test_lvl],
})

print("\nMSE / MAE / RMSE / R² table (VAR, 25 cells, diff vs level):")
print(metrics_table_var.to_string(index=False))



===== VAR baseline on 25 sampled US grid cells (non-seasonal part only, full time range) =====
Total time steps (raw): 548
ts_df_raw shape: (548, 25)

After seasonal differencing (lag = 12):
Total time steps (seasonally differenced): 536
ts_df shape (seasonally differenced): (536, 25)
Train range: 1981-01-01 00:30:00 → 2012-04-01 00:30:00
Val   range: 2012-04-01 00:30:00 → 2018-12-01 00:30:00
Test  range: 2018-12-01 00:30:00 → 2025-08-01 00:30:00

[Scaled series lengths (non-seasonal)]
T_train = 375 T_val = 80 T_test = 81

[Order selection by manual VAR(p) grid on non-seasonal series, with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK: AIC=-37.6862, BIC=-30.8659, HQIC=-34.9782, FPE=4.3206e-17
  p=2 OK: AIC=-37.1536, BIC=-23.7487, HQIC=-31.8307, FPE=7.6405e-17
  p=3 OK: AIC=-37.3464, BIC=-17.3306, HQIC=-29.3976, FPE=6.9820e-17
  p=4 OK: AIC=-37.2888, BIC=-10.6354, HQIC=-26.7029, FPE=9.0907e-17
  p=5 OK: AIC=-37.8329, BIC=-4.5152, HQIC=-24.5988, FPE=7.5324e-17
  p=6 OK: AIC=-39.0454, BIC=0.

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


### VAR Baseline — 25 Sampled US Grid Cells (Full Time Range, Non-Seasonal Component)

**Data Summary**  
- Total time steps (raw): **548**  
- Grid cells: **25**  
- After seasonal differencing (lag = 12): **536** time steps  
- Train range: **1981-01-01 → 2012-04-01**  
- Validation range: **2012-04-01 → 2018-12-01**  
- Test range: **2018-12-01 → 2025-08-01**

**Scaled series lengths**  
- Train: **375**  
- Validation: **80**  
- Test: **81**

---

### Selected Order (Manual VAR(p) Grid Search on Non-Seasonal Series)

| lag |     AIC     |      BIC      |      HQIC      |        FPE        |
|----:|------------:|--------------:|----------------:|------------------:|
| 1   | -37.6862    | **-30.8659**  | -34.9782        | 4.3206e-17        |
| 2   | -37.1536    | -23.7487      | -31.8307        | 7.6405e-17        |
| 3   | -37.3464    | -17.3306      | -29.3976        | 6.9820e-17        |
| 4   | -37.2888    | -10.6354      | -26.7029        | 9.0907e-17        |
| 5   | -37.8329    | -4.5152       | -24.5988        | 7.5324e-17        |
| 6   | -39.0454    | 0.9634        | -23.1519        | 3.9400e-17        |
| 7   | -41.1188    | 5.6083        | -22.5546        | 1.1601e-17        |
| 8   | -43.6204    | 9.8522        | -22.3741        | 3.3044e-18        |
| 9   | -47.6492    | 12.5963       | -23.7093        | 3.5686e-19        |
| 10  | -53.8548    | 13.1914       | -27.2097        | 9.8964e-21        |
| 11  | -63.3315    | 10.5431       | -33.9697        | 3.7403e-23        |
| 12  | **-77.7387** | 2.9924       | **-45.6484**    | **9.4857e-27**    |

- Selected lag by **BIC**: **1**  
- Final VAR order: **VAR(1)**

---

### Performance Summary (Statsmodels VAR)

#### 1. Results on Seasonally Differenced Series

|       Model          | Split |   RMSE   |    MSE    |    MAE    |     R²     |
|:--------------------:|:-----:|:--------:|:---------:|:---------:|:----------:|
| VAR_nonseasonal_diff |  VAL  | 2.114997 | 4.473213  | 1.489037  | -0.009408  |
| VAR_nonseasonal_diff | TEST  | 2.016881 | 4.067808  | 1.436725  | -0.005920  |

#### 2. Results on Reconstructed ORIGINAL LEVEL Series

|          Model              | Split |   RMSE   |    MSE    |    MAE    |     R²     |
|:---------------------------:|:-----:|:--------:|:---------:|:---------:|:----------:|
| VAR_level_reconstructed     |  VAL  | 2.114997 | 4.473213  | 1.489037  | 0.897445   |
| VAR_level_reconstructed     | TEST  | 2.016881 | 4.067808  | 1.436725  | 0.918480   |

---

### Comparison Table (Diff vs Level)

|                Model                | Split |   RMSE   |    MSE    |    MAE    |     R²     |
|:-----------------------------------:|:-----:|:--------:|:---------:|:---------:|:----------:|
| VAR_nonseasonal_diff               |  VAL  | 2.114997 | 4.473213  | 1.489037  | -0.009408  |
| VAR_nonseasonal_diff               | TEST  | 2.016881 | 4.067808  | 1.436725  | -0.005920  |
| VAR_level_reconstructed            |  VAL  | 2.114997 | 4.473213  | 1.489037  | 0.897445   |
| VAR_level_reconstructed            | TEST  | 2.016881 | 4.067808  | 1.436725  | 0.918480   |

---


## Short time

In [24]:
# ===== VAR on the 25 sampled US grid cells (last 200 time steps, seasonal part handled manually) =====
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VAR baseline on 25 sampled US grid cells "
      "(non-seasonal part only, last 200 time steps) =====")

# --------------------------------------------------
# 0. 基本設定與「原始」ts_df_raw
# --------------------------------------------------
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 25
T_total_raw = y_sample.shape[1]

ts_df_raw = pd.DataFrame(
    y_sample.T,  # (T_raw, 25)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps (raw):", T_total_raw)
print("ts_df_raw shape:", ts_df_raw.shape)  # (T_raw, 25)

# --------------------------------------------------
# 1. 手動處理季節成分：先做季節差分（lag = 12）
# --------------------------------------------------
SEASONAL_LAG = 12  # 一年 12 個月

ts_df_full = ts_df_raw.diff(SEASONAL_LAG).dropna()  # (T_total_full, 25)
T_total_full = ts_df_full.shape[0]
print("\nAfter seasonal differencing (lag = 12):")
print("Total time steps (seasonally differenced, full):", T_total_full)
print("ts_df_full shape:", ts_df_full.shape)

# --------------------------------------------------
# 1.5 限制在季節差分後的最後 200 個時間點
# --------------------------------------------------
USE_T = 200  # 只用最後 200 期
if T_total_full < USE_T:
    raise ValueError(f"季節差分後的時間長度只有 {T_total_full} < {USE_T}，無法取最後 200 期。")

ts_df = ts_df_full.iloc[-USE_T:].copy()
T_total = ts_df.shape[0]
idx_time = ts_df.index

print(f"\nRestricted to last {USE_T} time steps after seasonal differencing:")
print("Total time steps (restricted):", T_total)
print("ts_df shape (restricted, seasonally differenced):", ts_df.shape)
print(f"Restricted time range: {idx_time[0]} → {idx_time[-1]}")

# --------------------------------------------------
# 2. 在「最後 200 期」上切 70 / 15 / 15（train / val / test）
# --------------------------------------------------
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

print(f"Train range (restricted): {idx_time[0]} → {split_1_time}")
print(f"Val   range (restricted): {split_1_time} → {split_2_time}")
print(f"Test  range (restricted): {split_2_time} → {idx_time[-1]}")

# --------------------------------------------------
# 3. 用「restricted train」統計量標準化
# --------------------------------------------------
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec

T_train = cut_train
T_val   = cut_val - cut_train
T_test  = T_total - cut_val

print("\n[Scaled series lengths (non-seasonal, restricted)]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（季節差分後、原單位），用來算 MSE/MAE/R²（non-seasonal, restricted）
vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T_total, 25)
val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val, 25)
test_true_raw = vals_all[cut_val:, :]             # (T_test, 25)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳「季節差分後」序列的反標準化值（restricted 範圍），shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 在「restricted train」上做 VAR(p) 選階
# ======================================================
print("\n[Order selection by manual VAR(p) grid on non-seasonal series "
      "(restricted last 200 steps), with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)（restricted），可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK (restricted): AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, "
              f"HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（restricted, {e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 在 table 中標出每個指標的最佳（最小）值 ----
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"]],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"]],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"]],
})

printed_table.loc[best_idx_aic,  "AIC"]  += "*"
printed_table.loc[best_idx_bic,  "BIC"]  += "*"
printed_table.loc[best_idx_hqic, "HQIC"] += "*"
printed_table.loc[best_idx_fpe,  "FPE"]  += "*"

print("\nInformation Criteria table (p = 1..{} on non-seasonal series, restricted):"
      .format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 用 BIC 選 lag
p_selected = int(ic_table.loc[best_idx_bic, "lag"])
print(f"\nSelected lag by BIC (restricted) = {p_selected}")
print(f"Final VAR ORDER (non-seasonal, restricted) = ({p_selected})")

# ======================================================
# 4. 用 statsmodels VAR(p_selected) 做預測（restricted, 季節差分 + 標準化後）
# ======================================================
print(f"\n[Step] VAR(p={p_selected}) via statsmodels on non-seasonal *scaled* series "
      "(restricted)")

# 4.1 建立 restricted train / val / test 的 scaled DataFrame
ts_df_scaled_full = ts_df_scaled.astype("float64").copy()

train_scaled_df = ts_df_scaled_full.iloc[:cut_train]
val_scaled_df   = ts_df_scaled_full.iloc[cut_train:cut_val]
test_scaled_df  = ts_df_scaled_full.iloc[cut_val:]

T_train = train_scaled_df.shape[0]
T_val   = val_scaled_df.shape[0]
T_test  = test_scaled_df.shape[0]

print("Check lengths from scaled DF (restricted):",
      "T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 4.2 train → predict VAL
print("\n[Step] VAR: train → predict VAL (non-seasonal, scaled, restricted)")

var_train = VAR(train_scaled_df)
res_train = var_train.fit(p_selected)

y0_val = train_scaled_df.values[-p_selected:, :]      # shape (p_selected, 25)
pred_val_scaled = res_train.forecast(y0_val, steps=T_val)  # (T_val, 25)

pred_val_raw = inverse_scale(pred_val_scaled)         # (T_val, 25)

# 4.3 train+val → predict TEST
print("[Step] VAR: train+val → predict TEST (non-seasonal, scaled, restricted)")

train_val_scaled_df = ts_df_scaled_full.iloc[:cut_val]
var_train_val = VAR(train_val_scaled_df)
res_train_val = var_train_val.fit(p_selected)

y0_test = train_val_scaled_df.values[-p_selected:, :]     # shape (p_selected, 25)
pred_test_scaled = res_train_val.forecast(y0_test, steps=T_test)  # (T_test, 25)

pred_test_raw = inverse_scale(pred_test_scaled)           # (T_test, 25)

# ======================================================
# 5. 計算 VAL / TEST 的 MSE / MAE / RMSE / R²（non-seasonal, restricted）
# ======================================================
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

mse_var_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_var_val  = mean_absolute_error(y_val_true, y_val_pred)
mse_var_test = mean_squared_error(y_test_true, y_test_pred)
mae_var_test = mean_absolute_error(y_test_true, y_test_pred)

rmse_var_val  = np.sqrt(mse_var_val)
rmse_var_test = np.sqrt(mse_var_test)

r2_var_val  = r2_score(y_val_true,  y_val_pred)
r2_var_test = r2_score(y_test_true, y_test_pred)

print("\n===== VAR RESULTS (25 sampled US cells, non-seasonal series, "
      "statsmodels VAR, restricted last 200 steps) =====")
print(f"VAL  RMSE: {rmse_var_val:.4f}, MSE: {mse_var_val:.4f}, "
      f"MAE: {mae_var_val:.4f}, R²: {r2_var_val:.4f}")
print(f"TEST RMSE: {rmse_var_test:.4f}, MSE: {mse_var_test:.4f}, "
      f"MAE: {mae_var_test:.4f}, R²: {r2_var_test:.4f}")

metrics_table_var_restricted = pd.DataFrame({
    "Model": ["VAR_nonseasonal_last200", "VAR_nonseasonal_last200"],
    "Split": ["VAL",                     "TEST"],
    "RMSE":  [rmse_var_val,  rmse_var_test],
    "MSE":   [mse_var_val,   mse_var_test],
    "MAE":   [mae_var_val,   mae_var_test],
    "R2":    [r2_var_val,    r2_var_test],
})

print("\nMSE / MAE / RMSE / R² table (VAR, 25 cells, non-seasonal part, last 200 steps):")
print(metrics_table_var_restricted.to_string(index=False))



===== VAR baseline on 25 sampled US grid cells (non-seasonal part only, last 200 time steps) =====
Total time steps (raw): 548
ts_df_raw shape: (548, 25)

After seasonal differencing (lag = 12):
Total time steps (seasonally differenced, full): 536
ts_df_full shape: (536, 25)

Restricted to last 200 time steps after seasonal differencing:
Total time steps (restricted): 200
ts_df shape (restricted, seasonally differenced): (200, 25)
Restricted time range: 2009-01-01 00:30:00 → 2025-08-01 00:30:00
Train range (restricted): 2009-01-01 00:30:00 → 2020-09-01 00:30:00
Val   range (restricted): 2020-09-01 00:30:00 → 2023-03-01 00:30:00
Test  range (restricted): 2023-03-01 00:30:00 → 2025-08-01 00:30:00

[Scaled series lengths (non-seasonal, restricted)]
T_train = 140 T_val = 30 T_test = 30

[Order selection by manual VAR(p) grid on non-seasonal series (restricted last 200 steps), with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK (restricted): AIC=-42.8853, BIC=-29.1629, HQIC=-37.3089, FPE=2.6519

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


### VAR Baseline — 25 Sampled US Grid Cells (Last 200 Time Steps, Non-Seasonal Component)

**Data Summary**  
- Total time steps (raw): **548**  
- Grid cells: **25**  
- After seasonal differencing: **536**  
- Restricted to last 200 time steps: **200**  
- Restricted time range: **2009-01-01 → 2025-08-01**

**Restricted split ranges**  
- Train: **2009-01-01 → 2020-09-01**  
- Validation: **2020-09-01 → 2023-03-01**  
- Test: **2023-03-01 → 2025-08-01**

**Scaled series lengths (restricted)**  
- Train: **140**  
- Validation: **30**  
- Test: **30**

---

### Selected Order (Manual VAR(p) Grid Search on Non-Seasonal Restricted Series)

| lag |     AIC     |      BIC      |      HQIC      |        FPE        |
|----:|------------:|--------------:|----------------:|------------------:|
| 1   | -42.8853    | **-29.1629**  | -37.3089        | 2.6519e-19        |
| 2   | -43.6938    | -16.6486      | -32.7033        | 2.6459e-19        |
| 3   | -51.2947    | -10.7986      | -34.8381        | 1.7911e-21        |
| 4   | **-71.7677** | -17.6908     | **-49.7922**    | **2.9671e-27**    |

- Selected lag by **BIC**: **1**  
- Final VAR order (restricted): **VAR(1)**

---

### Performance Summary (Statsmodels VAR on Restricted Non-Seasonal Series)

|             Model              | Split |   RMSE   |    MSE    |    MAE    |     R²     |
|:------------------------------:|:-----:|:--------:|:---------:|:---------:|:----------:|
| VAR_nonseasonal_last200       |  VAL  | 2.131759 | 4.544398  | 1.524983  | -0.021475  |
| VAR_nonseasonal_last200       | TEST  | 1.919667 | 3.685121  | 1.385605  | -0.021419  |

---
